# 02 — Exploratory data analysis (stage 3)

**Purpose.** Describe the data the corrected models are trained on, show every correction of the
rebuild with evidence, and give the descriptive basis for the stage-6 decisions. The submitted
report had no EDA chapter and the original EDA notebook did not survive.

**Inputs.** Stage-5 output (clock-corrected, source-harmonised and cleaned data), the raw files
(for the time-zone figure) and the stage-4 / clock audit outputs. Figures and tables are produced
by `python -m solarpv.eda`; this notebook reads them and interprets them.

**Rules.** EDA changes no data. Anything that could inform a modelling choice is computed on the
**train split only** and labelled so; whole-period overviews mark validation and test.
Plan: `docs/stage3_eda_plan.md`. Every operation is listed in `docs/data_processing_record.md`.

In [1]:
import json

import pandas as pd

from solarpv.config.paths import OUTPUTS_DIR

EDA = OUTPUTS_DIR / "03_eda"
manifest = json.loads((EDA / "eda_manifest.json").read_text(encoding="utf-8"))
tables = {p.stem: pd.read_csv(p) for p in sorted((EDA / "tables").glob("*.csv"))}
print("produced by commit", manifest["git_commit"])

produced by commit cc4946b


## 1. The data at a glance

![overview](../outputs/03_eda/figures/fig01_overview.png)

Two of the 25 inverters of the 40 MW Ketura Solar park (3,449 kW AC), one minute resolution,
September 2021 to May 2025, aggregated to 32,856 hourly rows. Daily energy follows the seasons
(about 20 MWh on clear summer days, 12–15 MWh in winter) with cloudy-day dips concentrated in
winter and spring. The 30-day maximum hourly power stays between about 2,500 and 3,000 kW for the
whole period, with no step that would indicate a change of equipment or capacity (assumption
A-16). The hatched area is the second data source of 2025 (F-18); dashed lines mark the two
logger-clock resyncs (F-15).

In [2]:
tables["table_splits"]

,split,start_utc,end_utc,hours,daytime_hours,daytime_valid_target,daytime_op_limited,daytime_valid_normal_operation,daytime_polled,daytime_mixed
0,train,2021-08-31 21:00:00+00:00,2024-02-01 00:00:00+00:00,21195,9698,9656,193,9464,0,0
1,val,2024-02-01 00:00:00+00:00,2024-10-01 00:00:00+00:00,5832,2902,2901,106,2795,0,0
2,test,2024-10-01 00:00:00+00:00,2025-05-31 21:00:00+00:00,5829,2580,2575,106,2469,1001,70
3,all,2021-08-31 21:00:00+00:00,2025-05-31 21:00:00+00:00,32856,15180,15132,405,14728,1001,70


The split is temporal and defined in UTC. Exact counts replace the approximate values of
the submitted report (train ~21,049 h there; 21,195 h here). The test period has 2,580 daytime hours
(sun above 5° at mid-hour, F-02), of which 106 are op-limited (inverter stop or curtailment,
D-22) and 1,001 come from the polled source (D-21). Validation and training contain no polled
data, so the second source never enters training.

## 2. Time base: the original read local time as UTC (F-01)

![time bug](../outputs/03_eda/figures/fig02_time_bug.png)

In [3]:
tables["table_time_bug"]

,month,reading,clear_days,median_centroid_minus_solar_noon_min
0,"winter (Dec 2021, UTC+2)",original (local stamps read as UTC),6,120.4
1,"winter (Dec 2021, UTC+2)","rebuild (true UTC, clock-corrected)",6,1.9
2,"summer (Jul 2022, UTC+3)",original (local stamps read as UTC),6,173.6
3,"summer (Jul 2022, UTC+3)","rebuild (true UTC, clock-corrected)",6,2.6


On six clear days of a winter and a summer training month, the power-weighted centre of
production lies **+120 min** (winter, UTC+2) and **+174 min** (summer, UTC+3) after solar noon
when the raw stamps are read as UTC, as the original pipeline did. With the rebuild's true UTC
and clock correction it lies **+1.9 and +2.6 min** after solar noon. The summer value is not
exactly +180 min because the logger clock was already about 9 min slow then (section 3). In the
original, solar geometry and clear-sky features were therefore 2–3 h out of phase with the
target.

## 3. Logger clocks (F-15, D-19)

![clocks](../outputs/03_eda/figures/fig03_clock.png)

The meteo and inverter loggers drifted up to 16 minutes slow, mostly in the warm months, and were
resynchronised independently: on 2023-05-09 only the meteo clock was reset; on 2024-01-21 it was
set to the (drifted) inverter clock. The 2025 data source has its own clock, about 3 minutes slow.
After the correction, every monthly residual lies within ±2 minutes (lower panel). The flat
segment of the minute-source offsets after February 2025 is the last estimate carried forward:
only a few hours still come from that source, and a direct measurement on 2025-04-29 suggests
about 1.5 minutes more drift (a stated limitation).

## 4. The second data source of 2025 (F-18)

![polled](../outputs/03_eda/figures/fig04_polled.png)

From 2025-02-26 each tag is an instantaneous snapshot taken every 15 minutes, on its own phase,
and repeated until the next one. Held values lag by half a poll period and make an hourly mean a
mean of four stale samples. The rebuild puts each snapshot back at its poll instant and
interpolates linearly. A simulation that applies the same sampling to real minute data of the
training period gives an hourly error of 2.6 % of capacity for the held values and 1.6 % for the
rebuild (0.9 % on clear days); fusing the six tags naively was worse and was rejected.

## 5. Cleaning (stage 5, D-22)

![cleaning](../outputs/03_eda/figures/fig05_cleaning.png)

Value rules replace wrong measurements; flags mark real plant events without changing values.
The communication freeze of 11–12 September 2021 repeated the last values for hours, including
about 1,300 kW in the middle of the night; the frozen run is removed. This figure also exposed
a 247 kW blend minute at sunrise when the data resumed, which led to a margin around frozen
runs (R3). Daily energy is unchanged on all days except the few with spikes or freezes
(right panel).

## 6. Which irradiance sensor is right? (F-19)

![sensors](../outputs/03_eda/figures/fig10_sensors.png)

The site has two in-plane irradiance sensors. Against modelled clear-sky irradiance on the panel
plane, the sensor used by the models (`irradiance`, meteo station st01) is stable in every month,
while the redundant pyranometer drops to exactly ×0.75 from 2023-05-28 to 2023-12-31 and on days
between December 2024 and April 2025 — a calibration-constant fault. The pyranometer is not a
model input; its faulty days are removed by rule R7. This confirms decision D-09.

At the clearest hours the in-plane sensor reads about 10–20 % above the Ineichen clear-sky model
with default turbidity; the scale of the clear-sky denominator is examined in stage 6 (D-11).

In [4]:
tables["table_sensors"].head(24)

,month,irradiance_over_clearsky_p90,pyranometer_over_clearsky_p90,pyranometer_over_irradiance_median,pyranometer_over_irradiance_after_cleaning
0,2021-09,1.149,1.150,0.999,0.999
1,2021-10,1.147,1.138,0.999,1.000
2,2021-11,1.162,1.159,1.000,1.000
3,2021-12,1.107,1.101,0.996,0.996
4,2022-01,1.214,1.192,0.995,0.996
5,2022-02,1.163,1.173,0.999,0.999
6,2022-03,1.140,1.130,1.000,1.000
7,2022-04,1.163,1.158,1.002,1.002
8,2022-05,1.179,1.170,0.987,0.989
9,2022-06,1.112,1.114,0.997,0.999


## 7. Production patterns (train only)

![diurnal](../outputs/03_eda/figures/fig06_diurnal.png)

Mean production by local hour and month: a long, flat summer plateau and a short winter day.
Production starts around 06:00–07:00 local time all year, as expected for a fixed, south-facing
array at 30° N.

![power vs irradiance](../outputs/03_eda/figures/fig07_power_irradiance.png)

In [5]:
tables["table_physics"]

,quantity,value,basis
0,relative temperature coefficient of AC [%/K],-0.4220,"train, 700-900 W/m2"
1,AC per irradiance at 25 degC [kW per W/m2],2.8682,"train, 700-900 W/m2"


Hourly AC power is close to linear in the in-plane irradiance; at high irradiance the
spread is explained by module temperature. On normal-operation hours with 700–900 W/m², AC per unit
irradiance falls by **0.42 % per kelvin** of module temperature, within the usual range for
crystalline silicon (about −0.3 to −0.5 %/K). Temperature is therefore a physically meaningful
input, and clear-sky irradiance alone cannot explain summer noon output.

![variability]({FIG}/fig08_variability.png)

In [6]:
tables["table_variability"]

,month,days,median_daily_index,p10_daily_index
0,1,89.0,0.862,0.540
1,2,46.0,0.867,0.523
2,3,57.0,0.821,0.564
3,4,48.0,0.821,0.683
4,5,60.0,0.816,0.710
5,6,59.0,0.774,0.691
6,7,62.0,0.783,0.755
7,8,60.0,0.761,0.722
8,9,82.0,0.805,0.736
9,10,87.0,0.815,0.659


The daily clear-sky index (daily AC energy over a clear-sky reference) is high and narrow
in summer (median about 0.77, few cloudy days) and wide in winter and spring, where most
forecasting error will come from. Day-to-day persistence is modest: the autocorrelation of the
daily index is 0.33 at one day and near zero from two days on. Yesterday's production carries
some information about tomorrow, older days carry little — consistent with the small weight of
long lags and with the value of weather forecasts (stage 6b).

## 8. Operational events

![operations](../outputs/03_eda/figures/fig09_operations.png)

Inverter stops and grid curtailment affect from a few up to about 45 daytime hours per month, with
curtailment frequent in spring 2024 (validation) and autumn 2024 (test). These hours stay in the
data with their flags: they are excluded from the training loss and from past-day features,
included in the main evaluation and excluded in a secondary one (D-22). Missing-target hours
come from communication freezes.

## 9. What this means for the next stages

- All geometry is now in phase with production (section 2); the clear-sky and solar-position
  features of stage 6 can be trusted.
- The irradiance sensor is reliable over the whole period (section 6); its lags are valid inputs.
- Module temperature has a clear physical effect (section 7) — keep its lags.
- Persistence beyond one day is weak (section 7) — horizon-aware rolling features (D-04) and NWP
  (stage 6b) matter more than long lags.
- The clear-sky reference scale and the denominator of k (GHI vs POA, D-11) are decided in stage
  6 on train only.
- The 2025 test months need the per-source reporting of D-21.

**Reproduce:** `python -m solarpv.eda` (figures, tables, manifest), then run this notebook.